# 🛠️ Workshop 2: Building with GenAI

> **From consuming AI to building real applications**

---

### 🔄 Quick Recap: Workshop 1

You learned:
- ✅ What Generative AI is (AI that creates new content)
- ✅ How Transformers work (tokens, attention, parameters)
- ✅ Text generation with GPT-2
- ✅ Prompt engineering basics

**Today you BUILD things.** 👇

---

### 📋 What You'll Build

| # | Section | What You'll Create | Time |
|:-:|---------|-------------------|:----:|
| 1️⃣ | Project Structure | Set up a real GenAI project scaffold | 10 min |
| 2️⃣ | Image Generation | Generate images with FLUX AI 🎨 | 25 min |
| 3️⃣ | Local vs Cloud Models | Understand when to use each 🖥️ | 10 min |
| 4️⃣ | RAG Pipeline | Build a semantic search + answer engine 📚 | 25 min |
| 5️⃣ | Build a Mini Agent | Create an AI that uses tools 🤖 | 15 min |
| 6️⃣ | Wrap-Up & Next Steps | Bridge to Workshop 3 | 5 min |


---

# 🏗️ Section 1: Professional GenAI Project Structure

## Moving Beyond Colab: Real-World AI Development

### Why Project Structure Matters

```
❌ BAD: One giant notebook with everything
✅ GOOD: Organized folders, modular code, configuration files
```

### 📁 Ideal GenAI Project Structure (VSCode)

```
my-genai-project/
│
├── 📄 README.md              # Project documentation
├── 📄 requirements.txt       # Python dependencies
├── 📄 .env                   # API keys (NEVER commit!)
├── 📄 .gitignore             # Files to ignore in git
│
├── 📁 src/                   # Source code
│   ├── __init__.py
│   ├── models/               # Model loading & inference
│   │   ├── text_generator.py
│   │   └── image_generator.py
│   ├── pipelines/            # RAG, agents, workflows
│   │   ├── rag_pipeline.py
│   │   └── agent.py
│   └── utils/                # Helper functions
│       ├── prompts.py
│       └── data_loader.py
│
├── 📁 config/                # Configuration files
│   ├── model_config.yaml
│   └── prompts.yaml
│
├── 📁 data/                  # Data files
│   ├── raw/
│   └── processed/
│
├── 📁 notebooks/             # Jupyter notebooks (experiments)
│   └── experiments.ipynb
│
├── 📁 outputs/               # Generated outputs
│   ├── images/
│   └── text/
│
└── 📁 tests/                 # Unit tests
    └── test_models.py
```

### 🔑 Key Files Explained

| File | Purpose |
|------|--------|
| `requirements.txt` | List all packages: `transformers==4.35.0` |
| `.env` | Store secrets: `OPENAI_API_KEY=sk-...` |
| `.gitignore` | Exclude: `.env`, `outputs/`, `__pycache__/` |
| `config/*.yaml` | Model params, prompts, settings |

### 💡 Pro Tips

1. **Never hardcode API keys** - Use `.env` + `python-dotenv`
2. **Virtual environments** - `python -m venv .venv`
3. **Separate concerns** - Models, pipelines, utils in different files
4. **Version control** - Use Git from day 1

In [ ]:
# 📁 Let's ACTUALLY create a project structure!
import os

project_root = '/content/my_genai_project'
dirs = [
    'src/models',
    'src/pipelines',
    'src/utils',
    'config',
    'data/raw',
    'data/processed',
    'outputs/images',
    'outputs/text',
    'tests',
]

for d in dirs:
    os.makedirs(f'{project_root}/{d}', exist_ok=True)

# Create essential files
files = {
    '.gitignore': '.env\n__pycache__/\noutputs/\n*.pyc\n.venv/\n',
    '.env': '# NEVER commit this file!\nAPI_KEY=your_key_here\n',
    'requirements.txt': 'transformers\nlangchain\nfaiss-cpu\npython-dotenv\n',
    'README.md': '# My GenAI Project\n\nBuilt during Workshop 2!\n',
}
for filename, content in files.items():
    with open(f'{project_root}/{filename}', 'w') as f:
        f.write(content)

# Show the result
print('📁 Your project structure:')
print('=' * 40)
for root, dirs_list, files_list in os.walk(project_root):
    level = root.replace(project_root, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}📂 {os.path.basename(root)}/')
    sub_indent = '  ' * (level + 1)
    for file in files_list:
        print(f'{sub_indent}📄 {file}')

print('\n' + '=' * 40)
print('✅ Professional project structure created!')
print('💡 In real projects, open this in VS Code and start coding.')


---

# 🎨 Section 2: Image Generation with AI

## How Diffusion Models Work

```
TRAINING:
Clean Image → Add Noise → Add More Noise → Pure Noise
   (AI learns to reverse this process)

GENERATION:
Pure Noise → Remove Some Noise → Remove More → Final Image!
   (AI predicts and removes noise step by step)
```

### Popular Models

| Model | Creator | Best For |
|-------|---------|----------|
| FLUX | Black Forest Labs | Speed + Quality |
| Stable Diffusion | Stability AI | Open source, customizable |
| DALL-E 3 | OpenAI | Prompt understanding |
| Midjourney | Midjourney | Artistic quality |

In [ ]:
# 📦 Install Required Libraries
print("🔧 Installing image generation libraries...")
!pip install -q gradio_client Pillow
print("✅ Installation complete!")

🔧 Installing image generation libraries...
✅ Installation complete!


In [ ]:
# 🎨 Connect to FLUX Image Generation
from gradio_client import Client
from IPython.display import display, Image as IPImage
import warnings
warnings.filterwarnings('ignore')

print("🔗 Connecting to FLUX API...")
try:
    image_client = Client("black-forest-labs/FLUX.2-klein-9B")
    print("✅ Connected to FLUX.2 Klein 9B!")
    IMAGE_API_READY = True
except Exception as e:
    print(f"⚠️ API connection issue: {e}")
    print("We'll continue with demonstrations.")
    IMAGE_API_READY = False

🔗 Connecting to FLUX API...
Loaded as API: https://black-forest-labs-flux-2-klein-9b.hf.space ✔
✅ Connected to FLUX.2 Klein 9B!


In [ ]:
# ✨ Generate Your First Image!
from PIL import Image
import os

# ✨ Generate Your First Image with FLUX.2 Klein 9B!
def generate_image(
    prompt,
    input_images=[],           # Input images for image-to-image or editing tasks
    mode_choice="Distilled (4 steps)",  # 'Distilled (4 steps)' for fast, 'Base (50 steps)' for quality
    seed=0,                    # Seed for reproducibility (0 = random behavior with randomize_seed)
    randomize_seed=True,       # Whether to randomize the seed each generation
    width=1024,                # Output image width (multiples of 8, max 1024)
    height=1024,               # Output image height (multiples of 8, max 1024)
    num_inference_steps=4,     # Denoising steps (4 for Distilled, 50 for Base)
    guidance_scale=1,          # CFG scale - how closely to follow the prompt (1-20)
    prompt_upsampling=False    # Enhance/expand the prompt automatically
):
    """
    Generate an image from a text prompt using FLUX.2 Klein 9B.
    """

    print(f"🎨 Generating: '{prompt[:50]}...'")
    print(f"📊 Mode: {mode_choice} | Steps: {num_inference_steps} | CFG: {guidance_scale}")
    print("⏳ This takes 10-60 seconds depending on mode...")

    try:
        result = image_client.predict(
            prompt=prompt,
            input_images=input_images,
            mode_choice=mode_choice,
            seed=seed,
            randomize_seed=randomize_seed,
            width=width,
            height=height,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            prompt_upsampling=prompt_upsampling,
            api_name="/generate"
        )
        print("✅ Image generated!")
        # result[0] is the image dict, result[1] is the seed used
        image_info = result[0]
        seed_used = result[1]
        print(f"🌱 Seed used: {seed_used}")
        return image_info['path'] if isinstance(image_info, dict) else image_info
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
# Test generation with FLUX.2 Klein 9B
image_path = generate_image(
    prompt="A futuristic robot teaching AI to college students, digital art, vibrant colors, detailed",
    guidance_scale=1,
    prompt_upsampling=False
)

if image_path:
    # Convert webp to png for display
    try:
        img = Image.open(image_path)
        png_path = image_path.replace('.webp', '.png')
        img.save(png_path, 'PNG')
        display(IPImage(filename=png_path))
        os.remove(image_path) # Clean up the original webp file
    except Exception as e:
        print(f"❌ Error converting or displaying image: {e}")

🎨 Generating: 'A futuristic robot teaching AI to college students...'
📊 Mode: Distilled (4 steps) | Steps: 4 | CFG: 1
⏳ This takes 10-60 seconds depending on mode...
❌ Error: You have exceeded your GPU quota (85s requested vs. 81s left). Try again in 21:05:06


### 🎛️ Hyperparameters Explained
Here's a comprehensive explanation of all the hyperparameters in the **FLUX.2-klein-9B** model:

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **prompt** | `str` | *Required* | The text description of the image you want to generate. Be descriptive! |
| **input_images** | `list` | `[]` | Optional list of input images for image-to-image generation or editing. Leave empty for pure text-to-image. |
| **mode_choice** | `str` | `"Distilled (4 steps)"` | Generation mode: `"Distilled (4 steps)"` (fast) or `"Base (50 steps)"` (high quality). |
| **seed** | `float` | `0` | Random seed for reproducibility. Same seed + prompt = same image. |
| **randomize_seed** | `bool` | `True` | When `True`, generates a new random seed each time (ignores seed value). |
| **width** | `float` | `1024` | Output image width in pixels (must be multiples of 8). |
| **height** | `float` | `1024` | Output image height in pixels (must be multiples of 8). |
| **num_inference_steps** | `float` | `4` | Number of denoising iterations. More = better quality, slower. Use 4 for Distilled, 50 for Base. |
| **guidance_scale** | `float` | `1` | Classifier-Free Guidance scale. Higher values = more prompt adherence (range: 1-20). |
| **prompt_upsampling** | `bool` | `False` | When `True`, automatically enhances/expands your prompt for better results. |

### 🎯 Prompt Engineering for Images

| Component | Example | Effect |
|-----------|---------|--------|
| **Subject** | "A majestic lion" | What to generate |
| **Style** | "oil painting", "cyberpunk" | Visual style |
| **Details** | "golden mane, intense eyes" | Specific features |
| **Lighting** | "sunset lighting", "neon glow" | Mood & atmosphere |
| **Quality** | "8K, highly detailed" | Output quality |


💡 Usage Tips
1. For fast demos: Use mode_choice="Distilled (4 steps)" with guidance_scale=1
2. For high quality: Use mode_choice="Base (50 steps)" with guidance_scale=3-7
3. For reproducible results: Set randomize_seed=False and specify a seed value
4. For creative exploration: Use prompt_upsampling=True to let AI enhance your prompts


In [ ]:
# 🎮 YOUR TURN: Create Your Own Image!
#================================================
# 👇 EDIT THIS PROMPT
#================================================
my_prompt = "A magical library floating in space with glowing ancient books, fantasy art style, purple nebula background, cinematic lighting"
#================================================

my_image = generate_image(my_prompt)


if my_image:
    # Convert webp to png for display
    try:
        img = Image.open(my_image)
        png_path = my_image.replace('.webp', '.png')
        img.save(png_path, 'PNG')
        display(IPImage(filename=png_path))
        os.remove(my_image) # Clean up the original webp file
    except Exception as e:
        print(f"❌ Error converting or displaying image: {e}")

---

# 🖥️ Section 3: Local vs Cloud Models

## Why Run AI Locally?

| Aspect | Cloud API | Local Model |
|--------|-----------|-------------|
| Privacy | Data leaves your system | Data stays local |
| Cost | Pay per request | Free after download |
| Speed | Network latency | Instant response |
| Internet | Required | Not required |
| Customization | Limited | Full control |

## 🦙 Ollama: Easiest Way to Run Local LLMs

```bash
# Install: https://ollama.com
ollama pull qwen3:8b        # Download a model (~5GB)
ollama run qwen3:8b         # Chat interactively
```

### Popular Local Models (2026)

| Model | Size | Best For |
|-------|------|----------|
| Qwen 3 | 0.6B-235B | General purpose, multilingual |
| Llama 3.2 | 1B-90B | Meta's flagship |
| Gemma 3 | 1B-27B | Google's efficient model |
| Phi-4 | 14B | Microsoft's reasoning model |

Let's see how you'd use a local model from Python: 👇


In [ ]:
# 🖥️ Local Model Demo: The Python Pattern
# (Can't install Ollama in Colab, but here's the EXACT code pattern)

print('🖥️  LOCAL MODEL: The Ollama Python Pattern')
print('=' * 50)
print('''
# When you have Ollama installed locally:
import ollama

response = ollama.chat(
    model='qwen3:8b',
    messages=[{'role': 'user', 'content': 'Explain RAG in one sentence'}]
)
print(response['message']['content'])
''')

# Let's demonstrate the SAME pattern with cloud:
print('\n☁️  CLOUD MODEL: Google Colab AI (no key needed!)')
print('=' * 50)
try:
    from google.colab import ai
    result = ai.generate_text('Explain RAG (Retrieval Augmented Generation) in one sentence.')
    print(result)
except Exception as e:
    print(f'(Colab AI not available: {e})')
    print('Try in Google Colab, or use: pip install google-genai')

print('\n💡 KEY POINT: Same question, same pattern.')
print('   Local = ollama.chat()   |   Cloud = ai.generate_text()')
print('   In Workshop 3, you\'ll use BOTH!')


---

# 📚 Section 4: RAG Pipeline

## What is RAG? (Retrieval-Augmented Generation)

**Problem**: LLMs have knowledge cutoffs and don't know YOUR data.

**Solution**: RAG = Retrieve relevant info → Augment the prompt → Generate response

```
┌─────────────┐    ┌──────────────┐    ┌─────────────┐
│ Your Docs   │───▶│ Vector Store │───▶│ Retrieve    │
│ PDFs, etc   │    │ (Embeddings) │    │ Top-K Docs  │
└─────────────┘    └──────────────┘    └──────┬──────┘
                                              │
                                              ▼
┌─────────────┐    ┌──────────────┐    ┌─────────────┐
│ Answer!     │◀───│ LLM Generate │◀───│ Context +   │
│             │    │              │    │ Question    │
└─────────────┘    └──────────────┘    └─────────────┘
```

In [ ]:
# 📦 Install RAG Dependencies
print("🔧 Installing RAG libraries...")
!pip install -q langchain langchain-community sentence-transformers faiss-cpu
print("✅ RAG libraries installed!")

In [ ]:
# 📄 Step 1: Create Sample Documents
documents = [
    "Generative AI creates new content like text, images, and music using deep learning models.",
    "Transformers use attention mechanisms to process sequences and understand context.",
    "RAG combines retrieval systems with language models for accurate, grounded responses.",
    "Fine-tuning adapts pre-trained models to specific tasks using domain-specific data.",
    "AI agents can plan, use tools, and maintain memory to complete complex tasks autonomously.",
    "Vector databases store embeddings for fast similarity search in RAG applications.",
    "Prompt engineering is the art of crafting inputs to get optimal outputs from AI models.",
    "Healthcare AI can summarize medical reports and generate synthetic patient data for research."
]

print(f"📄 Created {len(documents)} sample documents")
for i, doc in enumerate(documents[:3], 1):
    print(f"   {i}. {doc[:60]}...")

In [ ]:
# 🧮 Step 2: Create Embeddings & Vector Store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

print("🧮 Creating embeddings (this may take a minute)...")

# Initialize embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create documents
docs = [Document(page_content=text) for text in documents]

# Create vector store
vectorstore = FAISS.from_documents(docs, embeddings)

print("✅ Vector store created!")
print(f"   📊 Stored {len(documents)} document embeddings")

In [ ]:
# 🔍 Step 3: Retrieve Relevant Documents
def retrieve_context(query, k=3):
    """Retrieve the most relevant documents for a query."""
    results = vectorstore.similarity_search(query, k=k)
    return "\n".join([doc.page_content for doc in results])

# Test retrieval
test_query = "How does RAG work?"
context = retrieve_context(test_query)

print(f"🔍 Query: '{test_query}'")
print("\n📚 Retrieved Context:")
print("-" * 50)
print(context)
print("-" * 50)

In [ ]:
# 🤖 Step 4: Generate Answer with Context
# Using Gemini (cloud) with GPT-2 as fallback

question = "What is RAG and why is it useful?"
context = retrieve_context(question, k=2)

augmented_prompt = f"""Based on this information:
{context}

Answer this question concisely: {question}
Answer:"""

print(f"❓ Question: {question}\n")
print("📚 Retrieved Context:")
print("-" * 50)
print(context)
print("-" * 50)

# Try cloud model first (much better answers)
print("\n🤖 RAG Response:")
print("=" * 50)
try:
    from google.colab import ai
    answer = ai.generate_text(augmented_prompt)
    print("☁️  (Gemini):")
    print(answer)
except Exception:
    # Fallback: GPT-2 (basic but works offline)
    from transformers import pipeline
    generator = pipeline("text-generation", model="gpt2", device=-1)
    result = generator(augmented_prompt, max_length=200, do_sample=True, temperature=0.7)
    print("💻 (GPT-2 fallback — for better answers, run in Colab):")
    print(result[0]['generated_text'])

print("=" * 50)
print("\n🎉 CHECKPOINT: RAG Pipeline Complete!")
print("💡 You just built: Documents → Embeddings → Retrieval → Generation")


---

# 🤖 Section 5: Build a Mini AI Agent

## What are AI Agents?

**Agents** = LLMs that can **plan**, **use tools**, and **take actions** autonomously.

```
┌───────────────────────────────────────┐
│          THE REACT PATTERN             │
├───────────────────────────────────────┤
│  User Question                         │
│       ↓                                │
│  🧠 THINK: "I need weather + math"     │
│       ↓                                │
│  🔧 ACT: Call get_weather("Delhi")    │
│       ↓                                │
│  👀 OBSERVE: "34°C, Sunny"            │
│       ↓                                │
│  🔧 ACT: Call calculator("25 * 4")    │
│       ↓                                │
│  👀 OBSERVE: "100"                    │
│       ↓                                │
│  💬 ANSWER: Combine results           │
└───────────────────────────────────────┘
```

Let's build one! 👇


In [ ]:
# 🤖 Build a Mini Agent with Tools!
import random

# === DEFINE TOOLS ===
def calculator(expression):
    """Evaluate a math expression"""
    try:
        return str(eval(expression))
    except:
        return "Error: invalid expression"

def get_weather(city):
    """Get weather for a city (simulated)"""
    weather_db = {
        "Delhi": "34°C, Sunny",
        "Mumbai": "29°C, Humid",
        "Bangalore": "25°C, Pleasant",
        "Chennai": "31°C, Hot",
    }
    return weather_db.get(city, f"{random.randint(20,35)}°C, Unknown")

def search_knowledge(query):
    """Search our RAG knowledge base (built in Section 4!)"""
    try:
        return retrieve_context(query, k=1)
    except:
        return "Knowledge base not loaded. Run Section 4 first."

tools = {
    "calculator": calculator,
    "get_weather": get_weather,
    "search_knowledge": search_knowledge,
}

# === AGENT LOOP ===
print('🤖 MINI AGENT v1.0')
print('=' * 50)
user_question = "What's the weather in Delhi, and what's 156 * 23? Also, what is RAG?"
print(f'👤 User: {user_question}\n')

# Step 1: Agent reasons about what tools to use
print('🧠 Agent thinking: "I need 3 things..."\n')

# Step 2: Execute tools
actions = [
    ("get_weather", "Delhi"),
    ("calculator", "156 * 23"),
    ("search_knowledge", "What is RAG?"),
]

results = {}
for tool_name, arg in actions:
    print(f'  🔧 Action: {tool_name}("{arg}")')
    result = tools[tool_name](arg)
    results[tool_name] = result
    print(f'  👀 Result: {result}\n')

# Step 3: Agent synthesizes
print('=' * 50)
print(f"🤖 Agent: Here's what I found:")
print(f"  • Weather in Delhi: {results['get_weather']}")
print(f"  • 156 × 23 = {results['calculator']}")
print(f"  • RAG: {results['search_knowledge'][:80]}...")
print('=' * 50)

print('\n💡 THIS is the ReAct pattern: Reason → Act → Observe → Repeat')
print('   Here WE chose the tools. In Workshop 3, the LLM does it AUTONOMOUSLY!')
print('\n🎉 CHECKPOINT: You just built an AI agent with 3 tools!')


In [ ]:
# 🎮 YOUR TURN: Add a new tool to the agent!
#=================================================
# 👇 Create your own tool function
#=================================================

def unit_converter(query):
    """Convert between units (add your own conversions!)"""
    conversions = {
        "km_to_miles": lambda x: x * 0.621371,
        "kg_to_lbs": lambda x: x * 2.20462,
        "celsius_to_fahrenheit": lambda x: (x * 9/5) + 32,
    }
    parts = query.split()
    value = float(parts[0])
    conversion = parts[1]
    if conversion in conversions:
        return f"{value} → {conversions[conversion](value):.2f}"
    return f"Unknown conversion: {conversion}"

# Test your tool
print('🔧 Testing unit_converter:')
print(f'  100 km_to_miles: {unit_converter("100 km_to_miles")}')
print(f'  37 celsius_to_fahrenheit: {unit_converter("37 celsius_to_fahrenheit")}')

# Add it to the agent
tools["unit_converter"] = unit_converter
print(f'\n✅ Agent now has {len(tools)} tools: {list(tools.keys())}')
#=================================================


---

# 🎓 What You Built Today

## Skills Unlocked 🔓

| Skill | What You Built | Key Code |
|-------|---------------|----------|
| **Project Structure** | Professional GenAI project scaffold | `os.makedirs()`, `.gitignore`, `.env` |
| **Image Generation** | AI images from text with FLUX | `gradio_client.Client.predict()` |
| **RAG Pipeline** | Semantic search + answer engine | `FAISS.from_documents()`, `similarity_search()` |
| **AI Agent** | Multi-tool agent with ReAct pattern | Tool functions + agent loop |

## Key Concepts

```
Diffusion: Pure Noise → Remove Noise Step by Step → Image!
RAG:       Documents → Embeddings → Vector Store → Retrieve → Generate
Agent:     Think → Act → Observe → Repeat
```

---

## 🚀 Hands-On Deep Dive: Agent Programming Guide

Want to build more complex agents? Open **Agent_Programming_Guide.ipynb** in this folder.
It walks you through building a complete ReAct agent with Ollama:
- 🛠️ Tool calling (calculator, weather, web search)
- 🧠 Memory (conversation history across turns)
- 🔄 Multi-tool chaining and self-reflection

---

## ➡️ Next: Workshop 3 — Healthcare AI Agent

In the next workshop, you'll:
- 🔬 **Analyze medical images** with multimodal AI (MedGemma)
- 📚 **Build a healthcare RAG** with real medical guidelines
- 🤖 **Deploy an autonomous agent** that manages a hospital
- ⚖️ **Detect and fix bias** in clinical AI models
- 🛡️ **Implement safety guardrails** for healthcare AI

**See you in [Workshop 3](../03_healthcare_ai_agent/)! 🚀**
